# Temporal probes

Label-free probes (no verification trials) of how much the stage-1 Conformers (see `src/README.md`'s "Temporal study") depend on frame order, on VoxCeleb1-O utterances:

- **Permutation sensitivity**: mean cosine distance between each utterance's OS embedding and its embedding with the order of `b`-frame blocks shuffled (`BS{b}`, `SS` for b = 1), as a function of the signal a block spans, `(b − 1) · 10 + 64` ms. Where the curve reaches 0 marks the time scale beyond which order no longer matters. The order-invariant model C stays at 0 throughout.
- **Attention distance**: attention-weighted mean |query − key| distance in ms per layer and head, on OS and SS input.

Needs the best checkpoints kept by the stage-1 runs (`training.keep_checkpoints`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch

ROOT = Path.cwd().parent  # notebook runs from doc/
sys.path.insert(0, str(ROOT))

from src.config import ExperimentConfig
from src.data.lazy_features import resolve_features
from src.data.voxceleb import VoxCelebCorpus
from src.evaluation.temporal_probes import attention_distances, permutation_sensitivity
from src.experiment import _checkpoint_path, _lazy_trial_utterances
from src.models.registry import build_model
from src.training.trainer import best_checkpoint_path

In [ ]:
models = {  # label -> (config the checkpoint was trained with, train strategy)
    "A (OS)": ("configs/conformer-voxceleb-stage1.yaml", "OS"),
    "B (RANDBS)": ("configs/conformer-voxceleb-stage1.yaml", "RANDBS"),
    "C (invariant)": ("configs/conformer-voxceleb-stage1-invariant.yaml", "OS"),
}
block_sizes = [1, 8, 16, 32, 64]  # SS, BS8, BS16, BS32, BS64
run_idx = 0
num_utterances = 1000      # VoxCeleb1-O utterances to probe
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def load(config_path, train_strategy):
    config = ExperimentConfig.from_yaml(ROOT / config_path)
    config.training.checkpoint_dir = str(ROOT / config.training.checkpoint_dir)
    model = build_model(config)
    checkpoint = torch.load(best_checkpoint_path(_checkpoint_path(config, train_strategy, run_idx)), map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    return config, model.to(device).eval()


loaded = {label: load(*entry) for label, entry in models.items()}
config = next(iter(loaded.values()))[0]
transformation = config.transformation
corpus = VoxCelebCorpus(config)
utterances = _lazy_trial_utterances(corpus, corpus.dev_trial_pairs, transformation)[:num_utterances]
segment_length = config.data.segment_length(transformation)

In [ ]:
strategies = ["SS" if b == 1 else f"BS{b}" for b in block_sizes]
block_ms = [(b - 1) * transformation.frame_step_s * 1000 + transformation.frame_length_s * 1000 for b in block_sizes]

fig = go.Figure()
for label, (_, model) in loaded.items():
    sensitivity = permutation_sensitivity(model, utterances, segment_length, strategies, device)
    fig.add_scatter(x=block_ms, y=[sensitivity[s] for s in strategies], mode="lines+markers", name=label)
fig.update_layout(xaxis_title="signal spanned by one shuffled block [ms]", yaxis_title="1 − cos(e_OS, e_BS)", xaxis_type="log")
fig.show()

In [ ]:
rng = np.random.default_rng(0)
segments = np.stack([resolve_features(features)[:segment_length] for features, _ in utterances[:200]])
shuffled = segments.copy()
for segment in shuffled:
    rng.shuffle(segment)

for label, (config, model) in loaded.items():
    step_ms = transformation.frame_step_s * 1000 * config.conformer.subsampling_time_stride ** 2
    for name, batch in (("OS", segments), ("SS", shuffled)):
        distances = attention_distances(model, torch.from_numpy(batch).float().unsqueeze(1).to(device), step_ms)
        print(f"{label:<14} {name}: mean attention distance [ms] per layer x head\n{np.round(distances).astype(int)}")